# Hackathon „Hack the Promoter" - Team Combat Hamsters

**iGEM Warsaw 2026**, 29 sierpnia 2026


## 5.1 Konfiguracja

Jedyna linia do podmiany to `KLUCZ`. Nagłówek `User-Agent` jest
potrzebny, bo klient korzysta z `urllib`.


In [3]:
KLUCZ = "hyppe_pWeWsYVTGMJWvwhK6pUWVF78CJ1xjHOd" # CHANGE IT
URL   = "https://hyppe.futura.foundation"

import json, statistics, urllib.error, urllib.request

UA = ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/128.0 Safari/537.36")


def wolaj(sciezka, dane=None, limit=900, prob=6):
    """(kod_http, odpowiedz). `dane` != None -> POST z JSON-em.
    """
    import time
    for nr in range(prob):
        z = urllib.request.Request(URL.rstrip('/') + sciezka)
        z.add_header('X-API-Key', KLUCZ)
        z.add_header('User-Agent', UA)
        if dane is not None:
            z.add_header('Content-Type', 'application/json')
            z.data = json.dumps(dane).encode()
        try:
            with urllib.request.urlopen(z, timeout=limit) as o:
                return o.status, json.loads(o.read().decode())
        except urllib.error.HTTPError as e:
            tresc = e.read().decode('utf-8', 'replace')
            ponawialne = (e.code == 503
                          or (e.code == 429 and sciezka != '/wgraj'))
            if not ponawialne or nr == prob - 1:
                return e.code, tresc
            czekaj = float(e.headers.get('Retry-After') or 0) or 0.4 * 2 ** nr
            time.sleep(min(czekaj, 8.0))
        except urllib.error.URLError as e:
            if nr == prob - 1:
                return 0, 'siec: %s' % e
            time.sleep(0.4 * 2 ** nr)


kod_, ja = wolaj('/me')
assert kod_ == 200, ('zly klucz albo brak User-Agent', ja)
print('druzyna    :', ja['druzyna'], '|', ja['uczestnik'])
print('klucz wazny:', ja['wazny_do'], '(%.1f h)' % (ja['wazny_jeszcze_s'] / 3600)
      if ja.get('wazny_jeszcze_s') else 'klucz bez terminu')
print('limity/min :', ja['limity_na_minute'])
print('wgranie za :', ja['zgloszenie_mozliwe_za_s'], 's')


druzyna    : druzyna_07 | druzyna_07_os1
klucz wazny: 2026-08-29 20:15 czasu lokalnego (5.0 h)
limity/min : {'/sedzia': 3000, '/nawigator/mapa': 3000, '/nawigator/edycje': 3000, 'inne': 600}
wgranie za : 0 s


## 5.2 Promotor wyjściowy

`GET /dziki` zwraca sekwencję 800 pz, jej długość i sumę kontrolną.


In [10]:
kod_, d = wolaj('/dziki')
assert kod_ == 200, d
DZIKI = d['sekwencja']
print('%s | gen %s | %s' % (d['nazwa'], d['gen'], d['genom']))
print('dlugosc %d bp | sha256[:12] = %s' % (d['dlugosc'], d['sha256_12']))
assert len(DZIKI) == 800

scaffold_9:210212-211011/rc | gen pks1 | P1 (Trichoderma atroviride)
dlugosc 800 bp | sha256[:12] = 6c091899e027


## a

In [12]:
from google.colab import drive
drive.mount('/content/drive')

PATH = '/content/drive/My Drive/HTP_2026'

Mounted at /content/drive


## 6.1 Funkcje pomocnicze do sortowania


In [16]:
import math

def compare_with_judge(seq_a_tuple, seq_b_tuple):
    """Compares two sequences using the Judge API. Returns -1 if A is stronger, 1 if B is stronger, 0 if tied/error."""
    seq_a, label_a = seq_a_tuple
    seq_b, label_b = seq_b_tuple

    # Truncate labels for the Judge API (max 40 characters)
    truncated_label_a = label_a[-40:] if len(label_a) > 40 else label_a
    truncated_label_b = label_b[-40:] if len(label_b) > 40 else label_b

    kod_, s = wolaj('/sedzia', {
        'a': seq_a,
        'b': seq_b,
        'nazwa_a': truncated_label_a,
        'nazwa_b': truncated_label_b
    })

    if kod_ != 200:
        print(f"Warning: Judge API error during comparison: {kod_}, {s}. Assuming sequences are equal.")
        return 0 # Treat as equal or handle error

    if s.get('silniejsza_idx') == 0:  # seq_a is stronger
        return -1
    elif s.get('silniejsza_idx') == 1: # seq_b is stronger
        return 1
    else:
        return 0 # They are equally strong, or an unexpected result

def merge_sort_with_judge(items):
    """Sorts a list of (sequence, label) tuples using merge sort and Judge API for comparisons."""
    if len(items) <= 1:
        return items

    mid = len(items) // 2
    left = merge_sort_with_judge(items[:mid])
    right = merge_sort_with_judge(items[mid:])

    return merge(left, right)

def merge(left, right):
    """Merges two sorted lists using compare_with_judge."""
    merged = []
    left_idx, right_idx = 0, 0

    while left_idx < len(left) and right_idx < len(right):
        # Compare left[left_idx] and right[right_idx] using the Judge
        comparison_result = compare_with_judge(left[left_idx], right[right_idx])

        if comparison_result == -1: # left item is stronger
            merged.append(left[left_idx])
            left_idx += 1
        elif comparison_result == 1: # right item is stronger
            merged.append(right[right_idx])
            right_idx += 1
        else: # They are equally strong or error, append both (order doesn't strictly matter here)
            merged.append(left[left_idx])
            merged.append(right[right_idx])
            left_idx += 1
            right_idx += 1

    merged.extend(left[left_idx:])
    merged.extend(right[right_idx:])
    return merged

## 6.2 Funkcje pomocnicze do heurystyk biologicznych

In [37]:
def apply_modification_step(current_pula, modification_logic_func, step_prefix, use_judge=True, mode=0):
    """General function to apply a modification and optionally judge the outcome."""
    new_pula = {}
    improvements_count = 0

    print(f"\n--- Applying step: {step_prefix} ---")
    for original_seq, label in current_pula.items():
        modified_seq, changes_made = modification_logic_func(original_seq, label, mode)

        if changes_made > 0:
            if use_judge:
                label_original_full = f"{label}_{step_prefix}_org"
                label_modified_full = f"{label}_{step_prefix}_mod"

                # Truncate labels for Judge API (max 40 characters)
                label_original_truncated = label_original_full[-40:]
                label_modified_truncated = label_modified_full[-40:]

                kod_, s = wolaj('/sedzia', {'a': original_seq, 'b': modified_seq,
                                            'nazwa_a': label_original_truncated, 'nazwa_b': label_modified_truncated})

                if kod_ == 200:
                    if s.get('silniejsza_idx') == 1: # Modified sequence is stronger
                        new_pula[modified_seq] = label_modified_full
                        improvements_count += 1
                    else: # Original is stronger or tied
                        new_pula[original_seq] = label_original_full
                else:
                    print(f"Warning: Judge API error for {label}. Keeping original. Error: {kod_}, {s}")
                    new_pula[original_seq] = label_original_full
            else:
                # If not using judge, just keep the modified sequence
                new_pula[modified_seq] = f"{label}_{step_prefix}_mod"
                improvements_count += 1
        else:
            # If no changes were made, keep the original sequence
            new_pula[original_seq] = label

    print(f"Total unique candidates after {step_prefix}: {len(new_pula)}")
    print(f"Number of sequences improved/modified by {step_prefix}: {improvements_count}")
    return new_pula

In [92]:
import re
import random

# Define Cre1 (repressor) and ACE2 (activator) patterns
CRE1_PATTERN = r'([CG][CT]GG[AG]G|C[TC]CC[GA][GC])'
ACE2_OPTIONS = ['GGCTAA', 'TTAGCC']

def replace_cre1_with_ace2(sequence_str, label, mode=0):
    """Replaces Cre1 motifs with a random ACE2 motif in a given sequence."""
    modified_seq_list = list(sequence_str)
    replacements_count = 0

    # Find all occurrences of Cre1 motifs
    # Iterate in reverse order to ensure that replacements don't affect indices of subsequent matches
    matches = list(re.finditer(CRE1_PATTERN, sequence_str))
    for match in reversed(matches):
        start, end = match.span()
        # Choose a random ACE2 motif
        replacement_motif = ACE2_OPTIONS[mode]

        # Replace the characters in the list
        modified_seq_list[start:end] = list(replacement_motif)
        replacements_count += 1

    return ''.join(modified_seq_list), replacements_count

In [86]:
import re
import random

# Define the new repressor pattern
REPRESSOR_PATTERN_NEW = r'(AGGCA|TGCCT)'

def replace_repressor_with_activator(sequence_str, label, mode=0):
    """Replaces specific repressor motifs with activator motifs and adjusts length."""
    modified_seq_list = list(sequence_str)
    replacements_count = 0

    # Find all occurrences of the new repressor motifs
    # Iterate in reverse order to ensure that replacements don't affect indices of subsequent matches
    matches = list(re.finditer(REPRESSOR_PATTERN_NEW, sequence_str))
    for match in reversed(matches):
        start, end = match.span() # end is exclusive, so match.span() gives (start, start + repressor_length)

        # Ensure there is a character *after* the repressor to remove
        # The character to remove is at the original `end` index.
        if end >= len(modified_seq_list):
            print(f"Warning: Repressor found at end of sequence {label}, cannot remove character after. Skipping this match at {start}.")
            continue

        replacement_motif = ACE2_OPTIONS[mode]

        # Replace the 5bp repressor with the 6bp activator
        # This temporarily increases the sequence length by 1
        modified_seq_list[start:end] = list(replacement_motif)

        # Remove the character immediately after the original repressor to maintain length.
        # The character originally at `end` is now at `end` (since the insert shifts elements).
        del modified_seq_list[end]

        replacements_count += 1

    return ''.join(modified_seq_list), replacements_count

In [51]:
TATA_END = "TATATATAGGTAGGCATGGTCCCGCTACTTGGCAGCAGTCGATGCAAATTATCATTCGGCACGCTCTGAATTCATTTTCTTTTACAGAATCATCACCGCC"
def ensure_tata_box_end(sequence_str, label, mode=0):
    # Replace ending, but only except for anything other than TATATATA (the rest should be original)
    modified_seq_list = list(sequence_str)
    modified_seq_list[-len(TATA_END):-len(TATA_END)+8] = list(TATA_END)[:8]
    return ''.join(modified_seq_list), 1

## 6.3 Funkcje pomocnicze do zapisywania do FASTY

In [7]:
def save_to_fasta(sequences_with_labels, filename, top_n=None):
    """Saves a list of (sequence, label) tuples to a FASTA file.

    Args:
        sequences_with_labels: A list of (sequence_string, label_string) tuples.
        filename: The path to the output FASTA file.
        top_n: If an integer, only the top_n sequences will be saved.
    """
    lines = []
    for i, (seq, label) in enumerate(sequences_with_labels):
        if top_n is not None and i >= top_n:
            break
        lines.append('>' + label)
        lines.append(seq)

    fasta_content = '\n'.join(lines)
    with open(filename, 'w') as f:
        f.write(fasta_content)
    print(f"Saved {min(len(sequences_with_labels), top_n) if top_n is not None else len(sequences_with_labels)} sequences to {filename}")


## 6.4 Funkcje pomocnicze do rekomendacji Nawigatora

In [8]:
def apply_navigator_recommendations(sequence, recommendations):
    """Applies Navigator's recommendations to a given sequence."""
    seq_list = list(sequence)
    for rec in recommendations:
        # Navigator positions are 1-based, Python lists are 0-based
        seq_list[rec['poz'] - 1] = rec['zmien_na']
    return ''.join(seq_list)

In [43]:
def get_navigator_recommendations_and_apply(sequence, label, mode=0):
    """Fetches Navigator recommendations and applies them to a sequence."""
    kod_nav, m_nav = wolaj('/nawigator/mapa', {'sekwencja': sequence, 'od': 0, 'ile': 800})
    if kod_nav != 200:
        print(f"Warning: Navigator API error for {label}. No recommendations applied. Error: {kod_nav}, {m_nav}")
        return sequence, 0

    recommendations = [w for w in m_nav['pozycje'] if w['zmien_na'] != '.']
    if recommendations:
        modified_seq = apply_navigator_recommendations(sequence, recommendations)
        return modified_seq, len(recommendations)
    else:
        return sequence, 0

## 6.4 Pętla algorytmu optymalizacyjnego

In [ ]:
NUM_ITERATIONS = 1
TOP_N_FOR_NEXT_ITERATION = 20 # Number of top sequences to carry over to the next iteration
TOTAL_FUZZED_SEQUENCES_PER_ITERATION = 100 # Total new fuzzed sequences to generate in each iteration
NUM_VARIANTS_PER_BASE = TOTAL_FUZZED_SEQUENCES_PER_ITERATION // TOP_N_FOR_NEXT_ITERATION

# Initialize the global pool of all unique sequences ever generated
all_sequences_ever_generated = {DZIKI: 'DZIKI_ORIGINAL'}

# Start the iterative process with the initial DZIKI sequence

current_pula_for_iteration, num_generated = generate_navigator_editions(DZIKI, "dziki",
                                                                    num_variants_to_produce=100)
ranked_this_iteration = list(current_pula_for_iteration.items())

for iteration in range(NUM_ITERATIONS):
    print(f"\n--- Starting Iteration {iteration + 1}/{NUM_ITERATIONS} ---")

    # Update current_pula_for_iteration with the top N sequences for the next round
    current_pula_for_iteration = {}
    for i in range(min(TOP_N_FOR_NEXT_ITERATION, len(ranked_this_iteration))):
        seq, label = ranked_this_iteration[i]
        current_pula_for_iteration[seq] = label

    print(f"Top {len(current_pula_for_iteration)} sequences selected for next iteration.")

    # Step 1: Generate new sequences using Navigator's edition proposals from the current best candidates
    next_round_pula = {}
    fuzzed_count_this_iteration = 0

    # Collect sequences from previous iteration for fuzzing
    bases_to_fuzz = list(current_pula_for_iteration.items())

    for seq_to_fuzz, label_to_fuzz in bases_to_fuzz:
        generated_seqs, num_generated = generate_navigator_editions(seq_to_fuzz, label_to_fuzz,
                                                                    num_variants_to_produce=NUM_VARIANTS_PER_BASE)
        for seq, label in generated_seqs.items():
            if seq not in next_round_pula: # Prevent duplicates if different bases generate same variant
                next_round_pula[seq] = f"{label_to_fuzz}_{label}"
                fuzzed_count_this_iteration += 1

    # Add original sequences from current_pula_for_iteration to next_round_pula
    # to ensure they are considered in the modification steps, but label them appropriately
    for seq, label in current_pula_for_iteration.items():
        next_round_pula[seq] = f"{label}_original" # To distinguish from new fuzzed variants

    print(f"Initial sequences carried over: {len(current_pula_for_iteration)}")
    print(f"Generated {fuzzed_count_this_iteration} new variants via Navigator Edits.")
    print(f"Total candidates for modification in this iteration: {len(next_round_pula)}")

    # Ensure unique sequences are tracked globally
    all_sequences_ever_generated.update(next_round_pula)

    # Step 2: Apply Cre1 to ACE2 replacement
    next_round_pula = apply_modification_step(next_round_pula, replace_cre1_with_ace2, 'cre1_to_a1', mode=0)
    all_sequences_ever_generated.update(next_round_pula)

    next_round_pula = apply_modification_step(next_round_pula, replace_cre1_with_ace2, 'cre1_to_a2', use_judge=False, mode=1)
    all_sequences_ever_generated.update(next_round_pula)

    # Step 3: Apply Repressor to Activator replacement
    next_round_pula = apply_modification_step(next_round_pula, replace_repressor_with_activator, 'r_to_a1', mode=0)
    all_sequences_ever_generated.update(next_round_pula)

    next_round_pula = apply_modification_step(next_round_pula, replace_repressor_with_activator, 'r_to_a2', use_judge=False, mode=1)
    all_sequences_ever_generated.update(next_round_pula)

    # next_round_pula = apply_modification_step(next_round_pula, ensure_tata_box_end, 'tata_box')
    # all_sequences_ever_generated.update(next_round_pula)

    # Step 4: Apply Navigator Recommendations
    # Using use_judge=False as recommendations are direct improvements, not comparisons.
    next_round_pula = apply_modification_step(next_round_pula, get_navigator_recommendations_and_apply, 'nav_rec', use_judge=False)
    all_sequences_ever_generated.update(next_round_pula)

    print(f"Iteration {iteration + 1} finished. Total unique candidates in this round: {len(next_round_pula)}")

    # Select top N candidates for the next iteration
    candidates_for_ranking = list(next_round_pula.items())

    # Perform sorting using the merge_sort_with_judge function
    ranked_this_iteration = merge_sort_with_judge(candidates_for_ranking)

print("\n--- Iterative Optimization Complete ---")
print(f"Total unique sequences collected across all iterations: {len(all_sequences_ever_generated)}")

# Final ranking of ALL unique sequences ever generated
# print("Starting final comprehensive ranking...")
final_ranked_sequences = ranked_this_iteration

In [98]:
import datetime, time
# Save with timestamp
save_to_fasta(all_sequences_ever_generated.items(), f'wynik_all_{datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S')}.fasta')

Saved 452 sequences to wynik_all_2026-08-29 17:50:48.fasta


In [ ]:
pula = final_ranked_sequences
wygrane = []
for i, (seq, etyk) in enumerate(pula, 1):
    kod_, s = wolaj('/sedzia', {'a': DZIKI, 'b': seq,
                                'nazwa_a': 'dziki', 'nazwa_b': etyk[-40:]})
    if kod_ != 200:
        print('%s: HTTP %s' % (etyk, kod_)); break
    if s['silniejsza_idx'] == 1:
        wygrane.append((etyk, seq))
print('przebily dzikiego: %d z %d sprawdzonych (%.0f%%)'
      % (len(wygrane), i, 100.0 * len(wygrane) / max(i, 1)))
print('UWAGA: wysylamy CALA pule (%d), nie tylko zwyciezcow.' % len(pula))
for etyk, _ in wygrane[:10]:
    print('  ', etyk)


## 6.5 Wgranie pliku FASTA

`POST /wgraj`, raz na 5 minut. Odpowiedź podaje statystyki
filtrowania i liczbę sekwencji, które weszły do oceny.


In [100]:
# Zwyciezcy Sedziego na poczatku pliku, potem reszta puli.
# Oceniane jest pierwsze 100 sekwencji.
LIMIT = 100
zwyciezcy = {s_ for _, s_ in wygrane}
do_wgrania = ([(e_, s_) for e_, s_ in wygrane]
              + [(e_, s_) for s_, e_ in pula if s_ not in zwyciezcy])
if len(do_wgrania) > LIMIT:
    print('pula %d -> przycieta do %d (limit regulaminowy)'
          % (len(do_wgrania), LIMIT))
    do_wgrania = do_wgrania[:LIMIT]
linie = []
for etyk, seq in do_wgrania:
    linie += ['>' + etyk, seq]
fasta = '\n'.join(linie)

# --- ZAPIS DO PLIKU ---
with open('wynik.fasta', 'w') as f:
    f.write(fasta)
# ----------------------

print('plik: %d sekwencji, %d znakow' % (len(do_wgrania), len(fasta)))

kod_, r = wolaj('/wgraj', {'fasta': fasta})
print(kod_)
if kod_ == 200:
    print('--- filtrowanie ---')
    for k2, v in r['filtrowanie'].items():
        print('   %-26s %s' % (k2, v))
    print('--- wynik ---')
    for k2 in ('ocenionych', 'dzielnik_top10', 'dzielnik_top100',
               'pozycja_top10', 'pozycja_top100', 'punkty_razem'):
        print('   %-26s %s' % (k2, r.get(k2)))
else:
    print(r)

pula 120 -> przycieta do 100 (limit regulaminowy)
plik: 100 sekwencji, 89097 znakow
200
--- filtrowanie ---
   n_w_pliku                  100
   n_dlugosc_800              100
   n_unikalnych               100
   n_alfabet_ok               100
   n_po_filtrze_N             100
   n_ocenianych               100
   odrzucone_dlugosc          0
   odrzucone_duplikaty        0
   odrzucone_alfabet          0
   odrzucone_N                0
   prog_N                     powyzej 80 z 800 zasad
   pominiete_powyzej_limitu   0
   limit_ocenianych           100
   identyfikatory_odrzuconych {}
   identyfikatory_pominietych []
--- wynik ---
   ocenionych                 100
   dzielnik_top10             10
   dzielnik_top100            100
   pozycja_top10              2
   pozycja_top100             4
   punkty_razem               16.0


## Analiza (WIP)

### Analiza zależności nukleotydów od warstw latentnych

In [67]:
import pandas as pd

layer_dependencies = []

# Assuming DZIKI is already defined and all_sequences_ever_generated contains all sequences.
# If all_sequences_ever_generated is empty for some reason, ensure DZIKI is added.
if not all_sequences_ever_generated and 'DZIKI' in globals():
    all_sequences_ever_generated = {DZIKI: 'DZIKI_ORIGINAL'}

# Ensure we iterate over a list of (sequence, label) tuples from the dictionary
for seq, label in list(all_sequences_ever_generated.items())[:10]:
    kod_, m = wolaj('/nawigator/mapa', {'sekwencja': seq, 'od': 0, 'ile': 800})
    if kod_ == 200:
        rozklad_warstw = m['rozklad_warstw']
        print(rozklad_warstw)
        total_nucleotides = 800 # Assuming fixed sequence length of 800bp

        # Convert counts to percentages
        percentages = {
            f'Layer_{layer}': (count / total_nucleotides) * 100
            for layer, count in rozklad_warstw.items()
        }
        percentages['Label'] = label # Add label for identification
        percentages['Sequence'] = seq # Add sequence for reference, though not displayed
        layer_dependencies.append(percentages)
    else:
        print(f"Warning: Could not get map for sequence {label}. Error: {kod_}, {m}")

# Convert to DataFrame for easier analysis and display
df_layer_dependencies = pd.DataFrame(layer_dependencies)

# Reorder columns to have Label first and then layers
if not df_layer_dependencies.empty:
    cols = ['Label'] + [col for col in df_layer_dependencies.columns if col.startswith('Layer_')]
    df_layer_dependencies = df_layer_dependencies[cols]

# Display the first few rows and descriptive statistics
print("Dependencies of nucleotides on latent layers (percentages):")
display(df_layer_dependencies.head())

print("\nDescriptive statistics for layer dependencies:")
display(df_layer_dependencies.describe())

{'0': 64, '1': 609, '2': 76, '3': 51}
{'0': 72, '1': 604, '2': 74, '3': 50}
{'0': 73, '1': 608, '2': 69, '3': 50}
{'0': 70, '1': 608, '2': 74, '3': 48}
{'0': 75, '1': 601, '2': 72, '3': 52}
{'0': 74, '1': 600, '2': 73, '3': 53}
{'0': 73, '1': 606, '2': 67, '3': 54}
{'0': 68, '1': 604, '2': 83, '3': 45}
{'0': 71, '1': 603, '2': 80, '3': 46}
{'0': 68, '1': 610, '2': 74, '3': 48}
Dependencies of nucleotides on latent layers (percentages):


,Label,Layer_0,Layer_1,Layer_2,Layer_3
0,DZIKI_ORIGINAL,8.000,76.125,9.500,6.375
1,edycje_L0_r1_1_edycje_L0_r1_1_cre1_to_a1_org,9.000,75.500,9.250,6.250
2,edycje_L0_r1_1_edycje_L0_r1_2_cre1_to_a1_org,9.125,76.000,8.625,6.250
3,edycje_L0_r1_1_edycje_L0_r1_3_cre1_to_a1_org_c...,8.750,76.000,9.250,6.000
4,edycje_L0_r1_2_edycje_L1_r1_1_cre1_to_a1_org,9.375,75.125,9.000,6.500



Descriptive statistics for layer dependencies:


,Layer_0,Layer_1,Layer_2,Layer_3
count,10.000000,10.000000,10.000000,10.000000
mean,8.850000,75.662500,9.275000,6.212500
std,0.419987,0.429187,0.588666,0.368226
min,8.000000,75.000000,8.375000,5.625000
25%,8.562500,75.406250,9.031250,6.000000
50%,8.937500,75.625000,9.250000,6.250000
75%,9.125000,76.000000,9.437500,6.468750
max,9.375000,76.250000,10.375000,6.750000
